**Imports**

In [4]:
import os
import torch
import numpy as np
import random
import numpy as np
import gymnasium
import ale_py

from utils import make_env, process_state
from dqn_agent import DQNAgent
from dqn_cnn_model import DQN_CNN_Model
from double_dqn_agent import DoubleDQNAgent

In [5]:
SEED = 23

torch.manual_seed(SEED)
torch.backends.cudnn.deterministic=True # https://discuss.pytorch.org/t/what-is-the-differenc-between-cudnn-deterministic-and-cudnn-benchmark/38054
torch.backends.cudnn.benchmark=True # https://discuss.pytorch.org/t/what-does-torch-backends-cudnn-benchmark-do/5936/4
np.random.seed(SEED)
random.seed(SEED)

In [6]:
DEVICE = "cpu"
if torch.cuda.is_available():
    DEVICE = "cuda"  
elif torch.backends.mps.is_available():
    DEVICE = "mps" 

In [7]:
GRAY_SCALE = True 
SCREEN_SIZE = 84 
NUM_STACKED_FRAMES = 4 
SKIP_FRAMES = 4 
ENV_NAME = "ALE/Breakout-v5" 

## Entrenamiento

In [8]:
#Hiperparámetros de entrenamiento del agente DQN
TOTAL_STEPS = 10_000_000
EPISODES = 10_000
STEPS_PER_EPISODE = 20_000

EPSILON_INI = 1
EPSILON_MIN = 0.05
EPSILON_ANNEAL_STEPS = 1_000_000

EPISODE_BLOCK = 100

BATCH_SIZE = 32
BUFFER_SIZE = 50_000

GAMMA = 0.995
LEARNING_RATE = 1e-5

In [9]:
env = make_env(ENV_NAME,
                video_folder='./videos/dqn_training',
                name_prefix="breakout",
                record_every=500,
                grayscale=GRAY_SCALE,
                screen_size=SCREEN_SIZE,
                stack_frames=NUM_STACKED_FRAMES,
                skip_frames=SKIP_FRAMES
                )

net = DQN_CNN_Model(env.observation_space.shape, env.action_space.n).to(DEVICE)

dqn_agent = DQNAgent(env, net, process_state, BUFFER_SIZE, BATCH_SIZE, LEARNING_RATE, GAMMA, 
                     epsilon_i=EPSILON_INI, epsilon_f=EPSILON_MIN, 
                     epsilon_anneal_steps=EPSILON_ANNEAL_STEPS, 
                     episode_block=EPISODE_BLOCK, device=DEVICE,with_priority=True)

#rewards_dqn = dqn_agent.train(EPISODES, STEPS_PER_EPISODE, TOTAL_STEPS)

env.close()

c:\Users\joaco\anaconda3\envs\obl_taller_ia\Lib\site-packages\gymnasium\wrappers\rendering.py:283: UserWarning: WARN: Overwriting existing videos at c:\Users\joaco\Documents\ort\entregas_taller_de_ia\obligatorio\videos\dqn_training folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(


In [10]:
dqn_prio = dqn_agent.train()

Training:   0%|          | 2/10000 [00:08<11:47:52,  4.25s/episode, reward=1.5, epsilon=1, steps=341]


KeyboardInterrupt: 

In [7]:
# --- 1. Vectorization Setup ---
NUM_ENVS = 50 # Number of parallel environments

# --- 2. Hyperparameters ---
TOTAL_STEPS = 10_000_000
BUFFER_SIZE = 50_000 
BATCH_SIZE = 32
GAMMA = 0.99
LEARNING_RATE = 1e-4

EPSILON_INI = 1.0
EPSILON_MIN = 0.05
EPSILON_ANNEAL_STEPS = 1_000_000
EPISODE_BLOCK = 100

def make_vec_env():
    # We will call your original make_env function from utils.py
    env = make_env(ENV_NAME,
                    video_folder=None,
                    grayscale=GRAY_SCALE,
                    screen_size=SCREEN_SIZE,
                    stack_frames=NUM_STACKED_FRAMES,
                    skip_frames=SKIP_FRAMES
                   )
    # --- START of ADDITION ---
    # We will add the statistics wrapper right here to be sure it's included.
    #env = gymnasium.wrappers.RecordEpisodeStatistics(env)
    # --- END of ADDITION ---
    return env


vec_env = gymnasium.vector.SyncVectorEnv([make_vec_env for _ in range(NUM_ENVS)])

# --- 5. Instantiate Agent and Model ---
net = DQN_CNN_Model(vec_env.single_observation_space.shape, vec_env.single_action_space.n).to(DEVICE)
#print(vec_env.observation_space.shape)

agent = DQNAgent(
    env=vec_env, 
    model=net, 
    obs_processing_func=process_state, 
    memory_buffer_size=BUFFER_SIZE, 
    batch_size=BATCH_SIZE, 
    learning_rate=LEARNING_RATE, 
    gamma=GAMMA, 
    epsilon_i=EPSILON_INI, 
    epsilon_f=EPSILON_MIN, 
    epsilon_anneal_steps=EPSILON_ANNEAL_STEPS, 
    episode_block=EPISODE_BLOCK, 
    device=DEVICE
)


# --- 6. Run the New Vectorized Training ---
# Call the new train_vec() method instead of train()
rewards_dqn = agent.train_vec()

# The environment is closed inside the train_vec method.

Vectorized Training: 100%|██████████| 1000000/1000000 [35:51<00:00, 464.89step/s, steps=1000000/1000000, avg_loss=0.036, epsilon=0.05, avg_reward=1.17]   
